# Task 2 (v2) — Multi-Class: CN vs MCI vs AD
## PeaceOfCode Hackathon | Track 2
**Best fit output: 94/94 steps, accuracy improving from ~0.46 → 0.63+**
Run all cells top to bottom on Google Colab.

## Step 0 — Install & Imports

In [ ]:
!pip install pydicom -q
print("Done")

In [ ]:
import os
import pydicom
import numpy as np
import cv2
import pandas as pd

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Extract MRI.zip

In [ ]:
import zipfile

zip_path     = "/content/drive/MyDrive/MRI.zip"   # adjust path if needed
extract_path = "/content/MRI"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction done")
print(os.listdir(extract_path)[:5])

## Step 3 — Upload MRI_metadata.csv

In [ ]:
from google.colab import files
uploaded = files.upload()   # upload MRI_metadata.csv here

## Step 4 — Load Metadata (CN=0, MCI=1, AD=2)

In [ ]:
df = pd.read_csv("MRI_metadata.csv")
print("Columns:", df.columns.tolist())
print()
print("Class distribution:")
print(df['Group'].value_counts())

# 3-class label map
label_map = {
    "CN":  0,
    "MCI": 1,
    "AD":  2
}
print()
print("Label map:", label_map)

## Step 5 — Load MRI Images
Takes **first 5 DICOM slices** per patient.
Handles both `/content/MRI` and `/content/MRI/MRI` folder structures.

In [ ]:
X = []
y = []

# handle both folder structures after unzip
base_path = "/content/MRI"
if os.path.exists("/content/MRI/MRI"):
    base_path = "/content/MRI/MRI"

print("Using base path:", base_path)
print("Total patient folders:", len(os.listdir(base_path)))

for patient in os.listdir(base_path):
    patient_path = os.path.join(base_path, patient)

    if not os.path.isdir(patient_path):
        continue

    # get label for this patient
    row = df[df["Subject"] == patient]
    if len(row) == 0:
        continue

    group = row["Group"].values[0]
    if group not in label_map:
        continue

    slices = []

    for root, dirs, files in os.walk(patient_path):
        for file in files:
            if file.endswith(".dcm"):
                path = os.path.join(root, file)
                try:
                    dicom = pydicom.dcmread(path)
                    img   = dicom.pixel_array.astype(np.float32)

                    img_max = np.max(img)
                    if img_max == 0:
                        continue
                    img = img / img_max   # normalize to [0,1]

                    img = cv2.resize(img, (128, 128))
                    slices.append(img)
                except:
                    continue

    # take first 5 slices per patient
    for img in slices[:5]:
        X.append(img)
        y.append(label_map[group])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print("X shape:", X.shape)
print("y distribution:", np.unique(y, return_counts=True))
print("  CN={}, MCI={}, AD={}".format(
    np.sum(y==0), np.sum(y==1), np.sum(y==2)
))

## Step 6 — Grayscale → RGB

In [ ]:
# CNN expects 3-channel input
X = X[..., np.newaxis]          # (N, 128, 128, 1)
X = np.repeat(X, 3, axis=-1)   # (N, 128, 128, 3)

print("After RGB conversion:", X.shape)

## Step 7 — One-Hot Encode Labels

In [ ]:
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, num_classes=3)

print("y_cat shape:", y_cat.shape)
print("Sample label (y / y_cat):", y[0], "→", y_cat[0])

## Step 8 — Train / Val Split (80/20, stratified)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y     # stratify on original integer labels
)

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)

## Step 9 — Data Augmentation

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=5,
    zoom_range=0.05
)

datagen.fit(X_train)
print("Augmentation ready")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)
print("Callbacks ready")

## Step 10 — Build Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.4),

    Dense(3, activation='softmax')   # 3 output classes
])

model.summary()

In [ ]:
model.compile(
    optimizer=Adam(0.0002),
    loss='categorical_crossentropy',   # matches to_categorical labels
    metrics=['accuracy']
)
print("Model compiled")

## Step 11 — Train

In [ ]:
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=8),
    validation_data=(X_val, y_val),
    epochs=15,
    callbacks=[early_stop],
    verbose=1
)

## Step 12 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy Curve')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss Curve')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.savefig("/content/Task2_v2_curves.png", dpi=150)
plt.show()
print("Saved!")

## Step 13 — Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, ConfusionMatrixDisplay

y_pred_probs = model.predict(X_val)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = np.argmax(y_val,        axis=1)

bal_acc = balanced_accuracy_score(y_true, y_pred)
cm      = confusion_matrix(y_true, y_pred)
report  = classification_report(y_true, y_pred, target_names=["CN", "MCI", "AD"])

print("=" * 45)
print("  TASK 2 (v2) — EVALUATION REPORT")
print("=" * 45)
print(f"  Balanced Accuracy: {bal_acc:.4f}")
print("=" * 45)
print()
print("Confusion Matrix:")
print(cm)
print()
print("Classification Report:")
print(report)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["CN", "MCI", "AD"])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
plt.title(f"Confusion Matrix\nBalanced Accuracy = {bal_acc:.3f}")
plt.tight_layout()
plt.savefig("/content/Task2_v2_confusion_matrix.png", dpi=150)
plt.show()

## Step 14 — Save Model & Results

In [ ]:
import shutil

# Save to Drive
model.save("/content/drive/MyDrive/Task2_v2_model.h5")
print("Model saved to Drive!")

drive_folder = "/content/drive/MyDrive/Task2_v2_Results"
os.makedirs(drive_folder, exist_ok=True)

shutil.copy("/content/Task2_v2_curves.png",          drive_folder)
shutil.copy("/content/Task2_v2_confusion_matrix.png", drive_folder)

# Save text report
report_text = f"""
=============================================
TASK 2 (v2) — EVALUATION REPORT
Model: CNN + categorical_crossentropy
Classes: CN=0, MCI=1, AD=2
=============================================
Balanced Accuracy : {bal_acc:.4f}

Confusion Matrix:
{cm}

Classification Report:
{report}
"""

with open(f"{drive_folder}/Task2_v2_report.txt", "w") as f:
    f.write(report_text)

print("All saved to:", drive_folder)
print(os.listdir(drive_folder))

In [ ]:
# Download to your computer
from google.colab import files
files.download("/content/Task2_v2_curves.png")
files.download("/content/Task2_v2_confusion_matrix.png")
print("Downloads started!")